# Hands-on with Python: Aprendizado Não-Supervisionado

- Este notebook contém uma aula prática sobre:
    - Algoritmos de Clusterização (K-means);
    - Redução de Dimensionalidade (PCA);
    - Preparação e Limpeza de Dados;
    - Visualização de Dados e Avaliação de Resultados.

- Usaremos os seguintes módulos escritos em Python:
    - Scikit-learn
    - Pandas
    - Matplotlib
    - Seaborn
    - NumPy

# 1. Importação de Bibliotecas para Análise Prática de Clusterização e PCA:

- `numpy, pandas:` Para manipulação e estruturação de dados, operações numéricas e tratamento de DataFrames.
- `matplotlib, seaborn:` Para visualização de dados e criação de gráficos informativos que ajudam a analisar os resultados.
- `sklearn.cluster (KMeans):` Implementação do algoritmo de clusterização K-means.
- `sklearn.decomposition (PCA):` Para aplicação da Análise de Componentes Principais (PCA) e redução de dimensionalidade.
- `sklearn.preprocessing (StandardScaler):` Normaliza os dados para uma escala uniforme, melhorando a precisão dos algoritmos.
- `sklearn.metrics (silhouette_score):` Avalia a qualidade dos clusters gerados pelo algoritmo de K-means.

Essas bibliotecas formam a base para realizar análises eficientes de dados e visualizações no aprendizado não-supervisionado.



In [ ]:
# Importando as bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


# 2. Carregamento e exploração inicial dos dados

### Conhecendo a base de dados:

O **Wine Quality** é um conjunto de dados utilizado para analisar a qualidade de vinhos portugueses, contendo 1.599 amostras de vinho tinto e 4.898 de vinho branco. Cada amostra possui 11 atributos físico-químicos e uma classificação de qualidade de 0 a 10, baseada em avaliações sensoriais. Os atributos incluem acidez, nível de açúcar e pH.

### Principais Características do Dataset

#### Atributos físico-químicos:
- Fixed Acidity (acidez fixa)
- Volatile Acidity (acidez volátil)
- Citric Acid (ácido cítrico)
- Residual Sugar (açúcar residual)
- Chlorides (cloretos)
- Free Sulfur Dioxide (dióxido de enxofre livre)
- Total Sulfur Dioxide (dióxido de enxofre total)
- Density (densidade)
- pH
- Sulphates (sulfatos)
- Alcohol (álcool)

#### Variável alvo:
- **Qualidade** (escala de 0 a 10, com base na avaliação sensorial).

O dataset pode ser obtido no UCI Machine Learning Repository, disponível para vinhos tintos e brancos.

**Fonte**: [UCI Machine Learning Repository - Wine Quality](https://archive.ics.uci.edu/ml/datasets/Wine+Quality)

In [ ]:
# Carregando o dataset Wine Quality
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine_df = pd.read_csv(url, delimiter=';')

# Exibindo as primeiras linhas do dataset
wine_df.head()

In [ ]:
wine_df.shape

In [ ]:
wine_df.info()

In [ ]:
wine_df.isnull().sum()

In [ ]:
wine_df.describe()

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(wine_df.corr(), annot=True)
plt.title('Correlação entre as colunas')
plt.show()

In [ ]:
X = wine_df.loc[:, ['fixed acidity', 'alcohol']]

In [ ]:
fig, ax = plt.subplots()
ax.scatter(X['fixed acidity'], X['alcohol'])

ax.set_xlabel('Acidez Fixa')
ax.set_ylabel('Álcool')
plt.title('Relação entre Acidez Fixa e Teor de Álcool')

plt.show()

- Utilizando o K-means

In [ ]:
kmeans2 = KMeans(n_clusters=2, random_state=0).fit(X)

In [ ]:
fig, ax = plt.subplots()

ax.scatter(X['fixed acidity'], X['alcohol'],c=kmeans2.labels_)
ax.set_xlabel('Acidez Fixa')
ax.set_ylabel('Álcool')
plt.title('Clusterização KMeans: Acidez Fixa e Teor de Álcool')
plt.show()

- E se usarmos k = 3? E k = 4? E 5?...

In [ ]:
# Utilizando k = 4
kmeans4 = KMeans(n_clusters=4, random_state=0).fit(X)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(X['fixed acidity'], X['alcohol'],c=kmeans4.labels_)
plt.show()

# 3. Pré-processamento dos Dados
### Separar características e rótulos

In [ ]:
wine_df = wine_df.drop('quality', axis=1)
wine_df.columns

# 4. Como determinar o valor ideal para K

### Método do Cotovelo

O `método do cotovelo` é uma técnica para determinar o número ideal de clusters em algoritmos de clusterização, como o KMeans. Ele envolve os seguintes passos:

1. `Cálculo da Inércia:` Aplicamos o KMeans para diferentes valores de \( k \) e calculamos a inércia, que mede a compactação dos clusters.

2. `Plotagem:` Os valores da inércia são plotados em um gráfico, com \( k \) no eixo x e a inércia no eixo y.

3. `Identificação do Cotovelo:` Analisamos o gráfico em busca de um ponto onde a redução da inércia começa a desacelerar, formando um "cotovelo". O valor de \( k \) nesse ponto é considerado ideal.

#### Implementação do Método do Cotovelo

In [ ]:
# Lista para armazenar a inércia
wss = []
for i in range(1, 10):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(wine_df)
    wss.append(kmeans.inertia_)

In [ ]:
plt.plot(range(1, 10), wss)
plt.title('O Método do Cotovelo')
plt.xlabel('Número de clusters')
plt.ylabel('Soma das distâncias quadradas')
plt.xticks(range(1, 10))
plt.grid(True)
plt.show()

Para melhorar a visualização do método do cotovelo, utilizaremos a biblioteca Yellowbrick. Ela ajuda a identificar o número ideal de clusters para o modelo KMeans aplicado ao conjunto de dados.

- Importação de KElbowVisualizer:

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

- Criação do modelo K-Means e configuração do KElbowVisualizer

In [ ]:
model = KMeans()
visualizer = KElbowVisualizer(model, k=(1,10), timings = False)
visualizer.fit(wine_df)
visualizer.show()

# 5. Avaliação da Qualidade de Clusters com Silhouette Score

Para avaliar a qualidade dos clusters formados pelo algoritmo KMeans será utilizada a `pontuação de silhueta(Silhouette Score)`. A pontuação de silhueta é uma métrica que quantifica quão bem cada ponto é agrupado em relação a outros clusters, variando de -1 a 1, onde valores mais altos indicam melhores agrupamentos.

In [ ]:
for i in range(2,10):
    kmeans = KMeans(n_clusters=i, max_iter=100)
    kmeans.fit(wine_df)
    score = silhouette_score(wine_df, kmeans.labels_)
    print("Para o cluster: {}, a pontuação da silhueta é: {}".format(i,score))

- Visualização da qualidade dos agrupamentos, onde coeficientes mais altos indicam melhor separação e coesão dos clusters formados.

In [ ]:
silhouette_coefficients = []
for i in range(2,10):
    kmeans = KMeans(n_clusters=i, max_iter=100)
    kmeans.fit(wine_df)
    score = silhouette_score(wine_df, kmeans.labels_)
    silhouette_coefficients.append(score)

plt.plot(range(2,10), silhouette_coefficients)
plt.xticks(range(2,10))
plt.xlabel("número de clusters")
plt.ylabel("Coeficiente de silhueta")
plt.show()

A visualização dos coeficientes de silhueta indica que o melhor número de clusters para o conjunto de dados é **3**.


# 6. Análise de componentes principais (PCA)

- Para treinar o modelo K-means com o número ideal de clusters identificado anteriormente, é fundamental reduzir a dimensionalidade do conjunto de dados.
- Isso ajuda a facilitar a visualização e a análise, enquanto preserva a maior parte da variância dos dados originais.
- A redução da dimensionalidade permite que os dados sejam representados de forma mais compreensível, melhorando a eficiência do algoritmo de clusterização.

In [ ]:
pca = PCA()
X = pca.fit_transform(wine_df)

In [ ]:
kmeans = KMeans(n_clusters=3)
label = kmeans.fit_predict(X)
unique_labels = np.unique(label)

In [ ]:
for i in unique_labels:
    plt.scatter(X[label==i,0], X[label==i,1], label=i, s=20)

plt.legend()
plt.title('Grupos de vinhos')
plt.show()

## Comparação do Teor Alcoólico entre Clusters

A seguir, é destacada uma análise comparativa do teor alcoólico entre os diferentes clusters. Afim de observar a distribuição do álcool em cada cluster, para identificar tendências como a mediana, a dispersão dos valores e a presença de outliers.

Essa análise ajudará a entender como o teor alcoólico varia entre os grupos e quais clusters apresentam maior ou menor concentração de álcool.

In [ ]:
wine_df['cluster'] = label

In [ ]:
cluster_analysis = wine_df.groupby('cluster')['alcohol'].describe()
print(cluster_analysis)

In [ ]:
sns.boxplot(x='cluster', y='alcohol', data=wine_df)
plt.title('Teor Alcoólico por Cluster')
plt.xlabel('Cluster')
plt.ylabel('Álcool')
plt.show()

### Histograma para comparar a distribuição de acidez fixa dos vinhos nos três clusters gerados
- O gráfico expõe as diferenças e auxilia para interpretação de como os clusters estão relacionados a características importantes, como acidez fixa, útil na classificação e análise dos vinhos.

In [ ]:
fig, ax = plt.subplots()

colors = ['blue', 'green', 'orange']
for i in range(3):
    cluster_data = X[:, 0][kmeans.labels_ == i]
    ax.hist(cluster_data, bins=15, alpha=0.5, label=f'Cluster {i+1}', color=colors[i])

ax.set_xlabel('Acidez Fixa')
ax.set_ylabel('Frequência')
plt.title('Distribuição de Acidez Fixa por Cluster')

plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()
